In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Fnotebooks%2FObject_detection_with_few_shot_learning%2Futility_pole_analysis_with_few_Shot_learning.ipynb?utm_source=cropped_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Object detection with few shot learning

This notebook demonstrates how to detect a trasformer installed on a pole from Imagery Insights using the Gemini 2.5 Flash model via Vertex AI.

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai google-cloud-storage Pillow pydantic

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'YOUR-PROJECT-ID'  # @param {type:"string"}
REGION = 'global'      # @param {type:"string"}

BIGQUERY_DATASET_ID = 'imagery_insights___us' # @param {type:"string"}
BIGQUERY_TABLE = "cropped_observations_latest" # @param {type:"string"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE" # @param {type:"string"}
LIMIT = 10 # @param {type:"integer"}

MODEL_ID = "gemini-3.6-flash" # @param {type:"string"}
# Reocmmend low temperature for structural/spatial tasks
MODEL_TEMPERATURE = 0.1 # @param {type:"number"}

GITHUB_REPOSITORY_NAME = 'googlemaps-samples/insights-samples' # @param {type:"string"}
GITHUB_BRANCH_NAME = 'main' # @param {type:"string"}


## Imports and Initializing Clients

In [ ]:
from io import BytesIO
from urllib.request import urlopen
import typing

from google.cloud import bigquery, storage
from google.colab import data_table
from google import genai
from google.genai.types import GenerateContentConfig
from IPython.display import display, Image
from PIL import Image
from pydantic import BaseModel, Field


genai_client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)
data_table.enable_dataframe_formatter()

## Fetch Image URIs from BigQuery

Next, we'll query a BigQuery table to get the GCS URIs of the images we want to classify.

In [ ]:
BIGQUERY_SQL_QUERY = f"""
SELECT
  *
FROM
  `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE}`
   WHERE asset_type = "{ASSET_TYPE}"
LIMIT {LIMIT};
"""

# Execute BigQuery Query
try:
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    query_response_data = [dict(row) for row in query_job]
    gcs_uris = [item.get("gcs_uri") for item in query_response_data if item.get("gcs_uri")]

    print(f"Successfully fetched {len(gcs_uris)} GCS URIs:")
    for uri in gcs_uris:
        print(uri)
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")

## Define Image Classification Function

This function takes a GCS URI and a prompt, then uses the Gemini 2.5 Flash model to generate a description of the image.

In [ ]:
def image_from_uri(uri: str) -> Image:
  if uri.startswith('gs://'):
      uri_parts = uri.replace("gs://", "").split("/", 1)
      bucket_name = uri_parts[0]
      blob_name = uri_parts[1]

      # Download the image bytes directly
      bucket = storage_client.bucket(bucket_name)
      blob = bucket.blob(blob_name)
      image_bytes = blob.download_as_bytes()
  else:
      with urlopen(uri) as response:
        image_bytes = response.read()

  return Image.open(BytesIO(image_bytes))

## Classify Images

Finally, we loop through the GCS URIs we fetched and pass them to our classification function along with a prompt.

In [ ]:
few_shot_uri_with_transformer = f"https://github.com/{GITHUB_REPOSITORY_NAME}/blob/{GITHUB_BRANCH_NAME}/street_view_insights/cropped/notebooks/Object_detection_with_few_shot_learning/images/with_attachments.jpeg?raw=true" # @param {type:"string"}
few_shot_image_with_transformer = image_from_uri(few_shot_uri_with_transformer)

few_shot_uri_no_attachments = f"https://github.com/{GITHUB_REPOSITORY_NAME}/blob/{GITHUB_BRANCH_NAME}/street_view_insights/cropped/notebooks/Object_detection_with_few_shot_learning/images/no_attachments.jpeg?raw=true" # @param {type:"string"}
few_shot_image_no_attachments = image_from_uri(few_shot_uri_no_attachments)

print("Few-shot example URIs and JSON outputs defined.")

print("With attachments")
display(few_shot_image_with_transformer)

print("No attachments")
display(few_shot_image_no_attachments)

In [ ]:
few_shot_json_with_transformer = """{
  "asset_type": "utility pole",
  "transformers": 1,
  "telephone_or_junction_boxes": 0,
  "additional_notes": ""
}"""

few_shot_json_no_attachments = """{
  "asset_type": "utility pole",
  "transformers": 0,
  "telephone_or_junction_boxes": 0,
  "additional_notes": ""
}"""


example_contents = [
  # Few-shot example 1
  "Example 1 (Utility pole with a transformer):",
  few_shot_image_with_transformer,
  "Expected Output:",
  few_shot_json_with_transformer,

  # Few-shot example 2
  "Example 2 (Utility pole without attachments):",
  few_shot_image_no_attachments,
  "Expected Output:",
  few_shot_image_no_attachments,
]

query_prompt = """
Instructions:

1. Analyze the provided image. If the image does not clearly show a utility pole, return: {{"error": "No utility pole detected in the image."}}
2. Detect and count the following:
    * Transformers
    * Telephone or junction boxes
"""

# Output schema
class QueryResult(BaseModel):
  asset_type: str
  transformers: int
  telephone_or_junction_boxes: int
  additional_notes: str


def process_and_classify(gcs_uri: str) -> typing.Tuple[str, Image.Image]:
    """
    Downloads and processes an image using few-shot inference
    """
    try:
        query_image = image_from_uri(gcs_uri)
        query_contents = [
            query_image,
            query_prompt,
        ]

        contents = example_contents + query_contents
        response = genai_client.models.generate_content(
            model=MODEL_ID,
            contents=contents,
            config=GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=QueryResult,
                temperature=MODEL_TEMPERATURE,
            ),
        )

        return response.text, query_image
    except Exception as e:
        print(f"Error classifying image from URI {gcs_uri}: {e}")
        return "Classification failed.", None

In [ ]:
if 'gcs_uris' in locals() and gcs_uris:
    for uri in gcs_uris:
        print(f"--- Classifying {uri} ---")
        classification, img = process_and_classify(uri)
        print(f"Result: {classification}\n")
        if img:
          display(img)
else:
    print("No GCS URIs were found to classify.")